In [1]:
## Load required keys from .env
import os
from pathlib import Path
from dotenv import load_dotenv
_ = load_dotenv('../.env')

SECRET_KEY = os.environ.get("LANGFUSE_SECRET_KEY")
PUBLIC_KEY = os.environ.get("LANGFUSE_PUBLIC_KEY")
BASE_URL = os.environ.get("LANGFUSE_BASE_URL")


In [2]:
# imports for working with traces in Langfuse
from langfuse_utils import *
import requests

# Authentication
import base64
auth = base64.b64encode(f"{PUBLIC_KEY}:{SECRET_KEY}".encode()).decode()
H = {"Authorization": f"Basic {auth}"}

In [3]:
## Filters for Langfuse
# NAME_FILTER = "LangGraph"     # or None


In [4]:
# imports for notebook features
# import ipywidgets as w

In [5]:

start = datetime.fromisoformat("2025-07-25T14:30:00+00:00")
end   = datetime.fromisoformat("2025-07-25T15:30:00+00:00")

params = {
    "fromTimestamp": iso_utc_plus(start),
    "toTimestamp": iso_utc_plus(end),
    "limit": LIMIT,
}
r = requests.get(f"{BASE_URL}/api/public/traces", headers=H, params=params, timeout=20)
print(r.status_code, r.text[:400])


200 {"data":[{"id":"0a3d6a938d216d42d8282ba1f55a7c64","projectId":"wri_lcl","name":"LangGraph","timestamp":"2025-07-25T15:24:34.262Z","environment":"default","tags":[],"bookmarked":false,"release":null,"version":null,"userId":null,"sessionId":null,"public":false,"input":{"messages":[{"content":"","additional_kwargs":{},"response_metadata":{},"type":"human","name":null,"id":null,"example":false},{"cont


## Time slots to fetch

In [ ]:
## where to store traces data
trace_root = "traces_data"
os.makedirs(trace_root, exist_ok=True)


In [ ]:
# Provide time slots in CENTRAL TIME (CT) 
slots = [
    ("Ana Benavides",         "2025-07-25 09:30"),
    ("Kuang Keng Kuek Ser",   "2025-07-29 07:00"),
    ("Willie",                "2025-07-29 16:00"),
    ("Berton Pakpahan",       "2025-08-01 09:00"),
]

# Set this appropriately
INTERVIEW_LENGTH_IN_MINUTES = 60

In [ ]:
for who, ct_start in slots:
    f_iso, t_iso = ct_window_iso(ct_start, INTERVIEW_LENGTH_IN_MINUTES)
    rows = fetch_window(f_iso, t_iso, BASE_URL, H)
    
    # client-side filter to avoid server 400s on unknown params
    # if NAME_FILTER: rows = [r for r in rows if r.get("name") == NAME_FILTER]

    slug = "".join(c.lower() if c.isalnum() else "_" for c in who)
    folder = os.path.join(trace_root, slug)

    save_jsonl(os.path.join(folder, "traces_list.jsonl"), rows)
    save_csv(os.path.join(folder, "traces_summary.csv"), [summarize(r) for r in rows])

    print(f"{who}: {ct_start} CT → "
      f"{datetime.fromisoformat(f_iso).strftime('%H:%M')} .. "
      f"{datetime.fromisoformat(t_iso).strftime('%H:%M')} UTC | {len(rows)} traces")

## List all saved traces

In [6]:
!tree traces_data/

traces_data/
├── ana_benavides__sgf_
│   ├── traces_list.jsonl
│   └── traces_summary.csv
├── berton_pakpahan__ifmn_
│   ├── traces_list.jsonl
│   └── traces_summary.csv
├── kuang_keng_kuek_ser__pulitzer_
│   ├── traces_list.jsonl
│   └── traces_summary.csv
└── willie__mongabay_
    ├── traces_list.jsonl
    └── traces_summary.csv

5 directories, 8 files


## Load saved traces in to dataframes

In [7]:
ROOT = Path("traces_data")
assert ROOT.exists(), f"Folder not found: {ROOT.resolve()}"

In [8]:
# Load all CSVs with interview label
trace_rows = []
jsonl_index = {}  # interview -> {trace_id: full_row}
for traces_dir in sorted(d for d in ROOT.iterdir() if d.is_dir()):
    label = traces_dir.name
    csv_path   = traces_dir / "traces_summary.csv"
    jsonl_path = traces_dir / "traces_list.jsonl"

    idf = safe_read_csv(csv_path)
    if idf.empty:
        continue
    idf["interview"] = label

    # Timestamps in CT for readability
    if "timestamp" in idf:
        idf["timestamp_ct"] = idf["timestamp"].dt.tz_convert(ZoneInfo("America/Chicago"))
        idf["date_ct"] = idf["timestamp_ct"].dt.date
        idf["minute_ct"] = idf["timestamp_ct"].dt.floor("min")
    trace_rows.append(idf)

    # Minimal JSONL index for drilldowns
    idx = {}
    for rec in read_jsonl(jsonl_path):
        tid = rec.get("id")
        if tid:
            idx[tid] = rec
    jsonl_index[label] = idx

df = pd.concat(trace_rows, ignore_index=True)

In [14]:

# derive some new columns for convenience 

df["timestamp_ct"] = pd.to_datetime(df["timestamp_ct"], errors="coerce")
df["duration_s"] = pd.to_numeric(df["duration_s"], errors="coerce")
df["totalCost"] = pd.to_numeric(df["totalCost"], errors="coerce")
cats = pd.CategoricalDtype(["SUCCESS","PARTIAL","ERROR","NO_TOOLS"], ordered=True)
df["outcome_status"] = df["outcome_status"].astype(cats)

# Convenience booleans
df["used_pick_aoi"] = df["tool_names"].fillna("").str.contains(r"\bpick-aoi\b")
df["used_pick_dataset"] = df["tool_names"].fillna("").str.contains(r"\bpick-dataset\b")
df["used_pull_data"] = df["tool_names"].fillna("").str.contains(r"\bpull-data\b")
df["has_aoi"] = df["selected_aoi"].notna() & (df["selected_aoi"].astype(str).str.len()>0)
df["has_dataset"] = df["selected_dataset"].notna() & (df["selected_dataset"].astype(str).str.len()>0)
df["apology_like"] = df["ai_output_trunc"].str.contains(r"apolog|unable to|failed to|cannot|can\'t|can't", case=False, na=False)


In [15]:
df.head(2).T

,0,1
ai_output_trunc,"I apologize, but I'm unable to pull the distur...",NaN
createdAt_utc,2025-07-25T15:25:12Z,2025-07-25T15:23:16Z
dataset_context_layer,"Classified drivers of DIST Alerts (aka LDACS),...",NaN
duration_bucket,>10s,0.5–1s
duration_s,34.911,0.72
end_date,2024-12-31,NaN
environment,default,default
final_ai_tokens_out,128,NaN
has_output,True,False
has_tool_failure,True,False


## Scoreboard! 

In [ ]:

scoreboard = (df.groupby("interview")
    .agg(traces=("id","count"),
         success=("outcome_status", lambda s: (s=="SUCCESS").sum()),
         partial=("outcome_status", lambda s: (s=="PARTIAL").sum()),
         error=("outcome_status", lambda s: (s=="ERROR").sum()),
         no_tools=("outcome_status", lambda s: (s=="NO_TOOLS").sum()),
         aoi_cov=("has_aoi","sum"),
         dataset_cov=("has_dataset","sum"),
         pick_aoi=("used_pick_aoi","sum"),
         pick_dataset=("used_pick_dataset","sum"),
         pull_data=("used_pull_data","sum"),
         med_dur_s=("duration_s","median"),
         p95_dur_s=("duration_s", p95),
         cost_total=("totalCost","sum"),
         cost_median=("totalCost","median"))
    .sort_values(["error","partial","success"], ascending=[False,False,True])
)
scoreboard["success_rate"] = (scoreboard["success"] / scoreboard["traces"]).round(3)
scoreboard["coverage_aoi_rate"] = (scoreboard["aoi_cov"] / scoreboard["traces"]).round(3)
scoreboard["coverage_dataset_rate"] = (scoreboard["dataset_cov"] / scoreboard["traces"]).round(3)
